# Hypothesis Test: Reasoning Strength Effect on Simulatability

This notebook performs **McNemar's test** comparing **None** vs **Low** (and **High**) reasoning levels for:
- GPT-5.2
- Claude Sonnet 4.5

## Why McNemar's Test?

McNemar's test is appropriate because:
1. **Paired data**: The same questions are answered under both conditions (None and Low reasoning)
2. **Binary outcome**: Predictor is either correct or incorrect
3. **Tests the difference in proportions**: Focuses on discordant pairs (correct under one condition but not the other)

In [10]:
import duckdb
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.contingency_tables import mcnemar

## Load the raw ITC data using DuckDB

In [11]:
def load_itc_data(filepath):
    """Load ITC parquet data using duckdb (handles nested types better)."""
    conn = duckdb.connect()
    df = conn.execute(f'''
        SELECT 
            original_question_idx,
            original_dataset,
            counterfactual_reference_response_answer,
            counterfactual_reference_response_model_info_thinking,
            counterfactual_predictor_response_with_explanation_answer
        FROM read_parquet('{filepath}')
    ''').df()
    conn.close()
    return df

# Load Sonnet and GPT data
df_sonnet = load_itc_data('../experiments/dewi_upload/sonnet_itc_predictions.parquet')
df_gpt = load_itc_data('../experiments/dewi_upload/gpt_5_2_itc_predictions.parquet')

# Also load extra file for GPT None
df_extra = load_itc_data('../experiments/dewi_upload/extra_three_ref_models_predictions.parquet')
df_gpt_none = df_extra[df_extra['counterfactual_reference_response_model_info_thinking'] == 'none'].copy()

print("=== Claude Sonnet 4.5 ===")
print(f"Total rows: {len(df_sonnet)}")
print(df_sonnet['counterfactual_reference_response_model_info_thinking'].value_counts())
print()
print("=== GPT-5.2 ===")
print(f"Total rows: {len(df_gpt)}")
print(df_gpt['counterfactual_reference_response_model_info_thinking'].value_counts())
print(f"\nGPT None from extra file: {len(df_gpt_none)} rows")

=== Claude Sonnet 4.5 ===
Total rows: 27995
counterfactual_reference_response_model_info_thinking
medium    7000
none      7000
high      6999
low       6996
Name: count, dtype: int64

=== GPT-5.2 ===
Total rows: 20670
counterfactual_reference_response_model_info_thinking
medium    6906
high      6888
low       6876
Name: count, dtype: int64

GPT None from extra file: 6925 rows


## Prepare paired data for McNemar's test

For each question, we need to determine:
- Was the predictor correct under **None** reasoning?
- Was the predictor correct under **Low** reasoning?

A predictor is "correct" if its answer matches the reference model's answer for that thinking level.

In [12]:
def prepare_paired_data(df, level_a, level_b):
    """
    Create paired data for McNemar's test.
    
    Returns a DataFrame with one row per question, containing:
    - correct_a: Whether predictor was correct under level_a
    - correct_b: Whether predictor was correct under level_b
    """
    # Filter to each level
    df_a = df[df['counterfactual_reference_response_model_info_thinking'] == level_a].copy()
    df_b = df[df['counterfactual_reference_response_model_info_thinking'] == level_b].copy()
    
    # Compute correctness
    df_a['correct'] = (df_a['counterfactual_predictor_response_with_explanation_answer'] == 
                       df_a['counterfactual_reference_response_answer'])
    df_b['correct'] = (df_b['counterfactual_predictor_response_with_explanation_answer'] == 
                       df_b['counterfactual_reference_response_answer'])
    
    # Merge on question index
    paired = df_a[['original_question_idx', 'original_dataset', 'correct']].merge(
        df_b[['original_question_idx', 'original_dataset', 'correct']],
        on=['original_question_idx', 'original_dataset'],
        suffixes=(f'_{level_a}', f'_{level_b}')
    )
    
    return paired

# Prepare paired data for Sonnet: None vs Low
paired_sonnet_none_low = prepare_paired_data(df_sonnet, 'none', 'low')
print(f"Sonnet None vs Low: {len(paired_sonnet_none_low)} paired observations")

# Prepare paired data for Sonnet: None vs High  
paired_sonnet_none_high = prepare_paired_data(df_sonnet, 'none', 'high')
print(f"Sonnet None vs High: {len(paired_sonnet_none_high)} paired observations")

Sonnet None vs Low: 26236 paired observations
Sonnet None vs High: 26246 paired observations


In [13]:
# For GPT, we need to combine the main file with the extra file for None
# The GPT ITC file doesn't have 'none' - it's in the extra file

# Combine GPT data
df_gpt_combined = pd.concat([df_gpt, df_gpt_none], ignore_index=True)
print("GPT combined thinking levels:")
print(df_gpt_combined['counterfactual_reference_response_model_info_thinking'].value_counts())

# Prepare paired data for GPT
paired_gpt_none_low = prepare_paired_data(df_gpt_combined, 'none', 'low')
print(f"\nGPT None vs Low: {len(paired_gpt_none_low)} paired observations")

paired_gpt_none_high = prepare_paired_data(df_gpt_combined, 'none', 'high')
print(f"GPT None vs High: {len(paired_gpt_none_high)} paired observations")

GPT combined thinking levels:
counterfactual_reference_response_model_info_thinking
none      6925
medium    6906
high      6888
low       6876
Name: count, dtype: int64

GPT None vs Low: 18540 paired observations
GPT None vs High: 18549 paired observations


## Build McNemar contingency tables

The 2x2 table for McNemar's test:

|                    | Low Correct | Low Incorrect |
|--------------------|-------------|---------------|
| **None Correct**   | a           | b             |
| **None Incorrect** | c           | d             |

McNemar's test focuses on the **discordant pairs** (b and c):
- b = Correct under None, Incorrect under Low (reasoning hurts)
- c = Incorrect under None, Correct under Low (reasoning helps)

H0: b = c (no difference)
H1: c > b (Low reasoning improves accuracy)

In [14]:
def build_mcnemar_table(paired_df, level_a, level_b):
    """
    Build 2x2 contingency table for McNemar's test.
    
    Returns:
        table: 2x2 numpy array [[a, b], [c, d]]
        Where:
        - a = both correct
        - b = correct under A, incorrect under B
        - c = incorrect under A, correct under B  
        - d = both incorrect
    """
    col_a = f'correct_{level_a}'
    col_b = f'correct_{level_b}'
    
    a = ((paired_df[col_a] == True) & (paired_df[col_b] == True)).sum()   # Both correct
    b = ((paired_df[col_a] == True) & (paired_df[col_b] == False)).sum()  # A correct, B incorrect
    c = ((paired_df[col_a] == False) & (paired_df[col_b] == True)).sum()  # A incorrect, B correct
    d = ((paired_df[col_a] == False) & (paired_df[col_b] == False)).sum() # Both incorrect
    
    table = np.array([[a, b], [c, d]])
    return table, {'a': a, 'b': b, 'c': c, 'd': d}

def run_mcnemar_test(paired_df, level_a, level_b, model_name):
    """
    Run McNemar's test and print results.
    """
    table, counts = build_mcnemar_table(paired_df, level_a, level_b)
    
    print(f"\n{'='*60}")
    print(f"{model_name}: {level_a.upper()} vs {level_b.upper()} reasoning")
    print(f"{'='*60}")
    
    # Print contingency table
    print(f"\nContingency Table:")
    print(f"                     | {level_b.capitalize():^12} | {level_b.capitalize():^12} |")
    print(f"                     | Correct      | Incorrect    |")
    print(f"-" * 55)
    print(f"{level_a.capitalize():^12} Correct   | {counts['a']:^12} | {counts['b']:^12} |")
    print(f"{level_a.capitalize():^12} Incorrect | {counts['c']:^12} | {counts['d']:^12} |")
    
    # Calculate accuracy for each condition
    n_total = counts['a'] + counts['b'] + counts['c'] + counts['d']
    acc_a = (counts['a'] + counts['b']) / n_total * 100
    acc_b = (counts['a'] + counts['c']) / n_total * 100
    
    print(f"\nAccuracy under {level_a}: {acc_a:.2f}%")
    print(f"Accuracy under {level_b}: {acc_b:.2f}%")
    print(f"Difference: {acc_b - acc_a:+.2f}%")
    
    # Discordant pairs
    print(f"\nDiscordant pairs:")
    print(f"  {level_a} correct, {level_b} incorrect (b): {counts['b']}")
    print(f"  {level_a} incorrect, {level_b} correct (c): {counts['c']}")
    
    # Run McNemar's test
    # Use exact test if discordant pairs < 25, otherwise chi-squared
    n_discordant = counts['b'] + counts['c']
    
    if n_discordant < 25:
        result = mcnemar(table, exact=True)
        test_type = "exact"
    else:
        result = mcnemar(table, exact=False, correction=True)
        test_type = "chi-squared (with continuity correction)"
    
    print(f"\nMcNemar's test ({test_type}):")
    print(f"  Statistic: {result.statistic:.3f}")
    print(f"  p-value (two-tailed): {result.pvalue:.4f}")
    
    # One-tailed p-value (testing if level_b > level_a)
    # If c > b, then level_b is better, and one-tailed p = two-tailed p / 2
    # If c < b, then level_b is worse, and one-tailed p = 1 - (two-tailed p / 2)
    if counts['c'] > counts['b']:
        p_one_tailed = result.pvalue / 2
        direction = f"{level_b} > {level_a}"
    else:
        p_one_tailed = 1 - result.pvalue / 2
        direction = f"{level_b} < {level_a}"
    
    print(f"  p-value (one-tailed, {direction}): {p_one_tailed:.4f}")
    
    # Significance
    if result.pvalue < 0.001:
        sig = "*** (p < 0.001)"
    elif result.pvalue < 0.01:
        sig = "** (p < 0.01)"
    elif result.pvalue < 0.05:
        sig = "* (p < 0.05)"
    else:
        sig = "n.s."
    
    print(f"\n  Result: {sig}")
    
    return {
        'model': model_name,
        'comparison': f'{level_a} vs {level_b}',
        'n_pairs': n_total,
        'acc_a': acc_a,
        'acc_b': acc_b,
        'diff': acc_b - acc_a,
        'b': counts['b'],
        'c': counts['c'],
        'statistic': result.statistic,
        'p_two_tailed': result.pvalue,
        'p_one_tailed': p_one_tailed,
        'significant_05': result.pvalue < 0.05
    }

## Run McNemar's tests

In [15]:
# Claude Sonnet 4.5
results = []

results.append(run_mcnemar_test(paired_sonnet_none_low, 'none', 'low', 'Claude Sonnet 4.5'))
results.append(run_mcnemar_test(paired_sonnet_none_high, 'none', 'high', 'Claude Sonnet 4.5'))


Claude Sonnet 4.5: NONE vs LOW reasoning

Contingency Table:
                     |     Low      |     Low      |
                     | Correct      | Incorrect    |
-------------------------------------------------------
    None     Correct   |    16774     |     3614     |
    None     Incorrect |     3848     |     2000     |

Accuracy under none: 77.71%
Accuracy under low: 78.60%
Difference: +0.89%

Discordant pairs:
  none correct, low incorrect (b): 3614
  none incorrect, low correct (c): 3848

McNemar's test (chi-squared (with continuity correction)):
  Statistic: 7.275
  p-value (two-tailed): 0.0070
  p-value (one-tailed, low > none): 0.0035

  Result: ** (p < 0.01)

Claude Sonnet 4.5: NONE vs HIGH reasoning

Contingency Table:
                     |     High     |     High     |
                     | Correct      | Incorrect    |
-------------------------------------------------------
    None     Correct   |    17059     |     3334     |
    None     Incorrect |     3931 

In [16]:
# GPT-5.2
results.append(run_mcnemar_test(paired_gpt_none_low, 'none', 'low', 'GPT-5.2'))
results.append(run_mcnemar_test(paired_gpt_none_high, 'none', 'high', 'GPT-5.2'))


GPT-5.2: NONE vs LOW reasoning

Contingency Table:
                     |     Low      |     Low      |
                     | Correct      | Incorrect    |
-------------------------------------------------------
    None     Correct   |    12634     |     2765     |
    None     Incorrect |     2227     |     914      |

Accuracy under none: 83.06%
Accuracy under low: 80.16%
Difference: -2.90%

Discordant pairs:
  none correct, low incorrect (b): 2765
  none incorrect, low correct (c): 2227

McNemar's test (chi-squared (with continuity correction)):
  Statistic: 57.766
  p-value (two-tailed): 0.0000
  p-value (one-tailed, low < none): 1.0000

  Result: *** (p < 0.001)

GPT-5.2: NONE vs HIGH reasoning

Contingency Table:
                     |     High     |     High     |
                     | Correct      | Incorrect    |
-------------------------------------------------------
    None     Correct   |    12400     |     3012     |
    None     Incorrect |     2165     |     972    

## Summary Table

In [17]:
summary_df = pd.DataFrame(results)
print("\n" + "="*80)
print("SUMMARY OF McNEMAR'S TESTS")
print("="*80)
print(summary_df[['model', 'comparison', 'n_pairs', 'diff', 'b', 'c', 'p_two_tailed', 'significant_05']].to_string(index=False))


SUMMARY OF McNEMAR'S TESTS
            model   comparison  n_pairs      diff    b    c  p_two_tailed  significant_05
Claude Sonnet 4.5  none vs low    26236  0.891904 3614 3848  6.990551e-03            True
Claude Sonnet 4.5 none vs high    26246  2.274632 3334 3931  2.701553e-12            True
          GPT-5.2  none vs low    18540 -2.901834 2765 2227  2.951949e-14            True
          GPT-5.2 none vs high    18549 -4.566284 3012 2165  6.428275e-32            True


## Suggested text for paper